# 🤖 PromptCanary — CI/CD Integration Guide

This notebook covers every integration pattern for running PromptCanary automatically:

1. **GitHub Actions** — scheduled weekly check + PR gating
2. **Multi-provider matrix** — test drift across OpenAI, Gemini, and local Ollama simultaneously
3. **Slack/webhook alerts** — notify your team on drift detection
4. **Programmatic exit codes** — using `--fail-on-drift` for CD gates
5. **Baseline promotion workflow** — safe baseline update process
6. **Cost-aware scheduling** — test cheap models frequently, expensive ones weekly

## 1 — GitHub Actions: Weekly Scheduled Check

The simplest integration. Add this to `.github/workflows/promptcanary.yml`:

In [ ]:
!pip install promptcanary

In [6]:
WEEKLY_WORKFLOW = """
name: PromptCanary Drift Check

on:
  schedule:
    - cron: "0 9 * * 1"    # Every Monday 9am UTC
  workflow_dispatch:        # Manual trigger from GitHub UI

jobs:
  canary:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install PromptCanary
        run: pip install promptcanary

      - name: Run canary (OpenAI GPT-5.5)
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: |
          promptcanary run \\
            --config canary.yaml \\
            --provider openai/gpt-5.5 \\
            --output-json results.json \\
            --output-md report.md

      - name: Compare to baseline
        run: |
          promptcanary compare \\
            --baseline baselines/gpt-5.5-latest.json \\
            --current results.json \\
            --output-md drift_report.md \\
            --fail-on-drift

      - name: Upload reports
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: promptcanary-reports
          path: "*.json, *.md, *.html"
"""
print(WEEKLY_WORKFLOW)


name: PromptCanary Drift Check

on:
  schedule:
    - cron: "0 9 * * 1"    # Every Monday 9am UTC
  workflow_dispatch:        # Manual trigger from GitHub UI

jobs:
  canary:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install PromptCanary
        run: pip install promptcanary

      - name: Run canary (OpenAI GPT-5.5)
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: |
          promptcanary run \
            --config canary.yaml \
            --provider openai/gpt-5.5 \
            --output-json results.json \
            --output-md report.md

      - name: Compare to baseline
        run: |
          promptcanary compare \
            --baseline baselines/gpt-5.5-latest.json \
            --current results.json \
            --output-md drift_report.md \
            --fail-on-drift

      - name

## 2 — Multi-Provider Matrix (2026 Models)

Test the same canary suite across multiple providers in parallel:

In [7]:
MATRIX_WORKFLOW = """
name: PromptCanary Multi-Provider Drift Check

on:
  schedule:
    - cron: "0 9 * * 1"

jobs:
  canary:
    name: Canary (${{ matrix.provider }})
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        include:
          # ── Cloud (paid) ────────────────────────────────────────────────
          - provider: openai/gpt-5.5
            baseline: baselines/openai-gpt-5.5-latest.json
            secret: OPENAI_API_KEY

          - provider: openai/gpt-5.4
            baseline: baselines/openai-gpt-5.4-latest.json
            secret: OPENAI_API_KEY

          - provider: anthropic/claude-sonnet-4-6
            baseline: baselines/claude-sonnet-4.6-latest.json
            secret: ANTHROPIC_API_KEY

          - provider: gemini/gemini-3.5-flash
            baseline: baselines/gemini-3.5-flash-latest.json
            secret: GEMINI_API_KEY

          - provider: gemini/gemini-3.1-flash-lite
            baseline: baselines/gemini-3.1-flash-lite-latest.json
            secret: GEMINI_API_KEY

          # ── Local via Ollama (free, no API key) ─────────────────────────
          - provider: ollama/qwen3.6:27b
            baseline: baselines/qwen3.6-27b-latest.json
            secret: ""

          - provider: ollama/deepseek-r1:14b
            baseline: baselines/deepseek-r1-14b-latest.json
            secret: ""

    steps:
      - uses: actions/checkout@v4

      - name: Set up Ollama (local models only)
        if: startsWith(matrix.provider, 'ollama/')
        run: |
          curl -fsSL https://ollama.ai/install.sh | sh
          ollama pull ${{ matrix.provider | replace('ollama/', '') }}

      - run: pip install promptcanary

      - name: Run and compare
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
          GEMINI_API_KEY: ${{ secrets.GEMINI_API_KEY }}
        run: |
          promptcanary run --provider "${{ matrix.provider }}" --output-json results.json
          promptcanary compare \\
            --baseline "${{ matrix.baseline }}" \\
            --current results.json \\
            --fail-on-drift
"""
print(MATRIX_WORKFLOW)


name: PromptCanary Multi-Provider Drift Check

on:
  schedule:
    - cron: "0 9 * * 1"

jobs:
  canary:
    name: Canary (${{ matrix.provider }})
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        include:
          # ── Cloud (paid) ────────────────────────────────────────────────
          - provider: openai/gpt-5.5
            baseline: baselines/openai-gpt-5.5-latest.json
            secret: OPENAI_API_KEY

          - provider: openai/gpt-5.4
            baseline: baselines/openai-gpt-5.4-latest.json
            secret: OPENAI_API_KEY

          - provider: anthropic/claude-sonnet-4-6
            baseline: baselines/claude-sonnet-4.6-latest.json
            secret: ANTHROPIC_API_KEY

          - provider: gemini/gemini-3.5-flash
            baseline: baselines/gemini-3.5-flash-latest.json
            secret: GEMINI_API_KEY

          - provider: gemini/gemini-3.1-flash-lite
            baseline: baselines/gemini-3.1-flash-lite-latest.json
      

## 3 — Provider Reference (June 2026)

Current recommended model strings for each provider:

In [8]:
PROVIDER_REFERENCE = {
    "OpenAI": {
        "flagship":  "openai/gpt-5.5",
        "balanced":  "openai/gpt-5.4",
        "fast":      "openai/gpt-5.4-mini",
        "preview":   "openai/gpt-5.6-terra",   # limited preview as of Jun 2026
        "env_var":   "OPENAI_API_KEY",
        "note":      "GPT-4o, o3 retired. Use GPT-5.x family.",
    },
    "Anthropic": {
        "flagship":  "anthropic/claude-opus-4-8",
        "balanced":  "anthropic/claude-sonnet-4-6",
        "env_var":   "ANTHROPIC_API_KEY",
        "note":      "claude-3-5-sonnet-20241022 is legacy. Use Claude 4 family.",
    },
    "Google Gemini": {
        "flagship":  "gemini/gemini-3.1-pro",
        "balanced":  "gemini/gemini-3.5-flash",
        "fast":      "gemini/gemini-3.1-flash-lite",
        "env_var":   "GEMINI_API_KEY",
        "note":      "Gemini 2.x family deprecated. Use Gemini 3.x.",
    },
    "xAI": {
        "flagship":  "xai/grok-4",
        "env_var":   "XAI_API_KEY",
        "note":      "Grok-beta retired. Use grok-4.",
    },
    "Ollama (local, free)": {
        "best_overall":   "ollama/qwen3.6:27b",       # 77.2% SWE-bench, Apache 2.0
        "best_coding":    "ollama/qwen3-coder:30b",   # 256K context, 3.3B active params
        "best_reasoning": "ollama/deepseek-r1:14b",   # chain-of-thought, 14B, MIT
        "best_small":     "ollama/gpt-oss:20b",       # OpenAI open-weight, 16GB RAM
        "best_chat":      "ollama/llama3.3:8b",       # 8GB RAM, fast
        "env_var":        "(none — local, no key required)",
        "note":           "Run `ollama pull <model>` first. Zero cost, full privacy.",
    },
}

for provider, info in PROVIDER_REFERENCE.items():
    print(f"\n{'─'*50}")
    print(f"  {provider}")
    print(f"{'─'*50}")
    for k, v in info.items():
        if k != 'note':
            print(f"  {k:15s}: {v}")
    print(f"  Note: {info.get('note', '')}")


──────────────────────────────────────────────────
  OpenAI
──────────────────────────────────────────────────
  flagship       : openai/gpt-5.5
  balanced       : openai/gpt-5.4
  fast           : openai/gpt-5.4-mini
  preview        : openai/gpt-5.6-terra
  env_var        : OPENAI_API_KEY
  Note: GPT-4o, o3 retired. Use GPT-5.x family.

──────────────────────────────────────────────────
  Anthropic
──────────────────────────────────────────────────
  flagship       : anthropic/claude-opus-4-8
  balanced       : anthropic/claude-sonnet-4-6
  env_var        : ANTHROPIC_API_KEY
  Note: claude-3-5-sonnet-20241022 is legacy. Use Claude 4 family.

──────────────────────────────────────────────────
  Google Gemini
──────────────────────────────────────────────────
  flagship       : gemini/gemini-3.1-pro
  balanced       : gemini/gemini-3.5-flash
  fast           : gemini/gemini-3.1-flash-lite
  env_var        : GEMINI_API_KEY
  Note: Gemini 2.x family deprecated. Use Gemini 3.x.

────────

## 4 — Programmatic Pipeline Integration

In [9]:
# Example: embed PromptCanary into a deployment pipeline
# Run this before deploying a new prompt template to production.

import sys, tempfile
from pathlib import Path

sys.path.insert(0, str(Path('.').parent))

from promptcanary import (
    CanarySuite, FileBaselineStore,
    CanaryPrompt, JsonValidityProbe,
    KeywordPresenceProbe, RefusalProbe, compare,
)
from promptcanary.core.models import LLMResponse, ProviderConfig, DriftSeverity
from promptcanary.providers.base import BaseLLMProvider


class MockDeploymentProvider(BaseLLMProvider):
    """Simulates a CI pre-deployment provider check."""
    def __init__(self, passing=True):
        super().__init__(ProviderConfig(model_id="openai/gpt-5.5"))
        self._passing = passing
    def complete(self, prompt, *, system_prompt=None):
        content = '{"status": "ok", "data": [1,2,3]}' if self._passing else 'Sure! Here is the data: status ok, data 1 2 3'
        return LLMResponse(prompt_id=prompt.id, provider_model_id="openai/gpt-5.5", content=content, finish_reason="stop", latency_ms=50.0)


def pre_deployment_canary_check(
    suite: CanarySuite,
    provider,
    baseline_path: Path,
    max_severity: DriftSeverity = DriftSeverity.LOW,
) -> bool:
    """
    Run a canary check before deployment.
    Returns True if safe to deploy, False if drift exceeds threshold.
    """
    result = suite.run(provider, show_progress=False)

    if not baseline_path.exists():
        print("⚠️  No baseline found — saving this run as baseline.")
        store = FileBaselineStore(baseline_path.parent)
        store.save(result)
        return True

    store = FileBaselineStore(baseline_path.parent)
    baseline = store.load_from_path(baseline_path)
    drift = compare(baseline, result)

    _SEVERITY_RANK = {DriftSeverity.NONE: 0, DriftSeverity.LOW: 1, DriftSeverity.MEDIUM: 2, DriftSeverity.HIGH: 3, DriftSeverity.CRITICAL: 4}
    is_safe = _SEVERITY_RANK[drift.severity] <= _SEVERITY_RANK[max_severity]

    print(drift.summary)
    print(f"Max allowed: {max_severity.value} | Detected: {drift.severity.value}")
    print(f"Deploy gate: {'✅ PASS' if is_safe else '❌ BLOCK'}")
    return is_safe


# ── Demo the deployment gate ──────────────────────────────────────────────────
suite = CanarySuite(
    name="deploy-gate-suite",
    prompts=[CanaryPrompt(id="p1", text="Return JSON status object.")],
    probes=[JsonValidityProbe(), RefusalProbe(expect_refusal=False)],
)

with tempfile.TemporaryDirectory() as tmp:
    baseline_path = Path(tmp) / 'baseline.json'

    # First run → creates baseline
    print("=== First run (creates baseline) ===")
    safe = pre_deployment_canary_check(suite, MockDeploymentProvider(passing=True), baseline_path)

    # Save baseline manually for demo
    from promptcanary.storage.file import FileBaselineStore
    store2 = FileBaselineStore(Path(tmp))
    result = suite.run(MockDeploymentProvider(passing=True), show_progress=False)
    snap = store2.save(result)
    import shutil
    files = list(Path(tmp).glob("*.json"))
    if files:
        shutil.copy(files[0], baseline_path)

    print("\n=== Second run (drifted — should block) ===")
    safe = pre_deployment_canary_check(suite, MockDeploymentProvider(passing=False), baseline_path)

=== First run (creates baseline) ===
⚠️  No baseline found — saving this run as baseline.

=== Second run (drifted — should block) ===
⚠️  CRITICAL drift in 'deploy-gate-suite': 1 regression(s) detected (score: 100.00% → 50.00%, Δ -50.00%).
Max allowed: low | Detected: critical
Deploy gate: ❌ BLOCK


## 5 — Cost-Aware Scheduling Strategy

Different models have different costs. This matrix shows how to schedule intelligently:

In [10]:
SCHEDULING_STRATEGY = [
    # (provider, cost_tier, schedule, rationale)
    ("ollama/qwen3.6:27b",         "free",     "0 * * * *",    "Hourly — zero cost, catches fast rollouts"),
    ("ollama/deepseek-r1:14b",     "free",     "0 * * * *",    "Hourly — reasoning model, local"),
    ("gemini/gemini-3.1-flash-lite","$0.003/k","0 */4 * * *",  "Every 4h — cheapest Gemini 3 model"),
    ("openai/gpt-5.4-mini",        "$0.001/k", "0 */6 * * *",  "Every 6h — cost-effective OpenAI"),
    ("gemini/gemini-3.5-flash",    "$0.015/k", "0 9 * * 1,4",  "Mon+Thu — balanced Gemini"),
    ("anthropic/claude-sonnet-4-6","$0.003/k", "0 9 * * 1",    "Weekly — Anthropic main model"),
    ("openai/gpt-5.5",             "$0.030/k", "0 9 * * 1",    "Weekly — OpenAI flagship"),
    ("gemini/gemini-3.1-pro",      "$0.012/k", "0 9 1 * *",    "Monthly — flagship reasoning only"),
    ("anthropic/claude-opus-4-8",  "$0.015/k", "0 9 1 * *",    "Monthly — premium tier"),
]

print(f"{'Provider':<38} {'Cost':<10} {'Schedule':<18} Rationale")
print("─" * 100)
for provider, cost, schedule, note in SCHEDULING_STRATEGY:
    print(f"{provider:<38} {cost:<10} {schedule:<18} {note}")

print()
print("Rule of thumb: free/cheap models → frequent; expensive models → weekly or monthly.")
print("Local Ollama models can run continuously as a zero-cost early-warning system.")

Provider                               Cost       Schedule           Rationale
────────────────────────────────────────────────────────────────────────────────────────────────────
ollama/qwen3.6:27b                     free       0 * * * *          Hourly — zero cost, catches fast rollouts
ollama/deepseek-r1:14b                 free       0 * * * *          Hourly — reasoning model, local
gemini/gemini-3.1-flash-lite           $0.003/k   0 */4 * * *        Every 4h — cheapest Gemini 3 model
openai/gpt-5.4-mini                    $0.001/k   0 */6 * * *        Every 6h — cost-effective OpenAI
gemini/gemini-3.5-flash                $0.015/k   0 9 * * 1,4        Mon+Thu — balanced Gemini
anthropic/claude-sonnet-4-6            $0.003/k   0 9 * * 1          Weekly — Anthropic main model
openai/gpt-5.5                         $0.030/k   0 9 * * 1          Weekly — OpenAI flagship
gemini/gemini-3.1-pro                  $0.012/k   0 9 1 * *          Monthly — flagship reasoning only
anthropic/c

## 6 — Baseline Promotion Workflow

When a provider intentionally changes behavior (new model version, deliberate format change),
you need to **promote** the new run as the accepted baseline. Best practices:

In [11]:
BASELINE_PROMOTION_WORKFLOW = """
# Baseline Promotion Workflow
# ────────────────────────────────────────────────────────────

# Step 1: Detect drift (happens automatically in CI)
promptcanary compare --provider openai/gpt-5.5 --fail-on-drift
# → EXIT CODE 1: drift detected

# Step 2: Review the drift report manually
promptcanary compare --provider openai/gpt-5.5 --output-html drift.html
open drift.html  # inspect what changed

# Step 3a: The change is ACCEPTABLE (intentional model upgrade)
# Promote: run again and save as the new baseline
promptcanary run \\
    --provider openai/gpt-5.5 \\
    --save-baseline \\
    --baseline-dir baselines/
# → commit the new baseline JSON to your repo
# → tag the commit: git tag baseline/gpt-5.5-2026-07-01

# Step 3b: The change is UNACCEPTABLE (unexpected regression)
# Pin the previous model version, open an incident
# Update your LiteLLMProvider call to use a versioned model string:
#   LiteLLMProvider("openai/gpt-5.4")  # pin until regression is fixed
"""
print(BASELINE_PROMOTION_WORKFLOW)


# Baseline Promotion Workflow
# ────────────────────────────────────────────────────────────

# Step 1: Detect drift (happens automatically in CI)
promptcanary compare --provider openai/gpt-5.5 --fail-on-drift
# → EXIT CODE 1: drift detected

# Step 2: Review the drift report manually
promptcanary compare --provider openai/gpt-5.5 --output-html drift.html
open drift.html  # inspect what changed

# Step 3a: The change is ACCEPTABLE (intentional model upgrade)
# Promote: run again and save as the new baseline
promptcanary run \
    --provider openai/gpt-5.5 \
    --save-baseline \
    --baseline-dir baselines/
# → commit the new baseline JSON to your repo
# → tag the commit: git tag baseline/gpt-5.5-2026-07-01

# Step 3b: The change is UNACCEPTABLE (unexpected regression)
# Pin the previous model version, open an incident
# Update your LiteLLMProvider call to use a versioned model string:
#   LiteLLMProvider("openai/gpt-5.4")  # pin until regression is fixed



---

## Summary Cheat Sheet

```bash
# First-time setup (any provider)
promptcanary init my-suite
promptcanary run --provider openai/gpt-5.5 --save-baseline

# Weekly CI check (GitHub Actions)
promptcanary run --provider openai/gpt-5.5 --output-json results.json
promptcanary compare --current results.json --fail-on-drift

# Free local alternative (no API key)
ollama pull qwen3.6:27b
promptcanary run --provider ollama/qwen3.6:27b --save-baseline
promptcanary compare --provider ollama/qwen3.6:27b --fail-on-drift

# Multi-provider comparison
for provider in openai/gpt-5.5 gemini/gemini-3.5-flash ollama/qwen3.6:27b; do
    promptcanary run --provider $provider --output-json results-${provider//\//-}.json
done
```

See the [GitHub Actions workflow](../.github/workflows/promptcanary.yml) for the full production setup.